# Lab: Words as Vectors

**Bluevision-ai Academy**
*Words as Vectors: From One-Hot to Geometry*

In lecture, one-hot encoding gave `dot(cat, feline) = 0` -- the Orthogonality Problem -- and the fix was Firth's distributional hypothesis, "you shall know a word by the company it keeps," engineered into Word2Vec's Skip-Gram network and paid off with `king - man + woman = queen`. This lab builds a Word2Vec model on real text, uses it, and then goes looking for where the famous demo breaks.

A hands-on build in five moves: **Problem** (what we have, what we want, why the obvious tool doesn't reach it) -> **Idea** (the reasoning move, in words, before any code) -> **Build** (the code that embodies exactly that idea) -> **Payoff** (what just happened, and what it means).

| Part | What you build | Builds on (lecture) |
|---|---|---|
| 1 -- Co-occurrence by Hand | Count a word's company directly; rebut `dot(cat, feline) = 0` with a real number | Slides 4-6 |
| 2 -- Word2Vec from Scratch | Skip-Gram with negative sampling, trained on real 19th-century text | Slides 9-13 |
| 3 -- Explore the Space | Nearest neighbours; pay off (or not) `king - man + woman` | Slides 14-15 |
| 4 -- Visualise with t-SNE | The five-cluster map, rebuilt on your own vectors, with the honesty check numbered | Slide 16 |
| Challenge | Break it: which analogies fail, and why | Slide 15 closing caution |
| Reflection | Five questions, answered in code comments | -- |

**A note on tooling.** The lecture's slide preview names `gensim` as the library for Part 2. This lab's build environment runs Python 3.14, and `gensim`'s compiled C extensions do not yet build on it (a real, current gensim/CPython incompatibility, not a workaround of convenience) -- so Part 2 implements Skip-Gram with negative sampling directly, matching Slides 10 and 13 mechanism-for-mechanism instead of calling a library. On Colab's own Python this would install fine; either way you end up owning the mechanism, which is the actual point.

**Dataset:** eighteen public-domain books from NLTK's Gutenberg corpus (the same corpus Session 01's lab used) -- Genesis-to-Revelation King James text, Shakespeare, Milton, Austen, Chesterton, Whitman. Real 19th-century-and-earlier English, not a modern web crawl, which turns out to matter a great deal by the time you reach the Challenge. **Estimated time:** ~2 hours, including a few minutes of model training. Runs top to bottom on Google Colab with no setup beyond the cell below.

## Setup

Colab already ships with most of this. The cell below makes sure versions are current, downloads the NLTK Gutenberg corpus (first run only), and loads the deck's own palette so these plots read like the ones you just watched.

In [1]:
# Quiet, idempotent installs -- safe to re-run.
!pip install -q --upgrade torch nltk scikit-learn plotly ipywidgets

import re
import time
from collections import Counter

import numpy as np
import nltk
import torch
import torch.nn as nn
import torch.nn.functional as F
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.manifold import TSNE

try:
    from google.colab import output
    output.enable_custom_widget_manager()
except ImportError:
    pass  # not running on Colab -- local Jupyter already works

try:
    nltk.data.find("corpora/gutenberg")
except LookupError:
    nltk.download("gutenberg", quiet=True)
from nltk.corpus import gutenberg

np.random.seed(0)
torch.manual_seed(0)
# Single-threaded + deterministic algorithms: CPU nn.Embedding backward uses a
# scatter-add whose accumulation order (and therefore its rounding) depends on
# thread scheduling otherwise -- without this, reruns of Part 2 change every
# number quoted in Parts 3-4 and the Challenge below.
torch.set_num_threads(1)
torch.use_deterministic_algorithms(True)

# The deck's own palette, so the plots here read like the plots you just watched.
GOLD, BLUE, TEAL, RED, DIM = "#b07d22", "#2563a8", "#148a7a", "#c0392b", "#7f8c9a"

## Part 1 -- Co-occurrence by Hand

**Problem.** Slides 4-5 gave one-hot encoding the full autopsy: `dot(cat, feline) = 0`, exactly the same zero you'd get for `cat` and `refrigerator`. Every word sits equally far from every other word -- the Orthogonality Problem. Slide 6's fix was a claim, not yet a computation: "you shall know a word by the company it keeps." Before training any neural network, let's test that claim in the most literal way possible -- count each word's company by hand, on a corpus small enough to read end to end, and see whether `cat` and `feline` come out as anything other than strangers.

**Ingredients.** Sixteen hand-built sentences, four little "topics" (cat/feline, dog/canine, banking, and houses), each topic using its two synonym words interchangeably across matching sentence templates -- exactly Firth's setup: no dictionary entry says `cat` and `feline` are related, only shared sentences do. We also strip a short stopword list (`the`, `a`, `my`, ...) before counting, for the same reason Part 2 will subsample frequent words later: filler words appear everywhere and would otherwise dominate every count.

In [2]:
sentences_toy = [
    "the cat sat on the mat",
    "the feline sat on the mat",
    "the cat chased the mouse",
    "the feline chased the small mouse",
    "my cat likes warm milk",
    "my feline likes warm milk",
    "the dog barked at the mailman",
    "the canine barked at the mailman",
    "the dog loves to run in the park",
    "the canine loves to run in the park",
    "she deposited money at the bank",
    "he applied for a loan at the bank",
    "the bank raised interest rates again",
    "the house on the hill has a red door",
    "they painted the house last summer",
    "we sold the house near the river",
]

STOPWORDS_TOY = {"the", "a", "my", "on", "at", "to", "in", "he", "she",
                  "we", "they", "for", "near", "last", "again"}

toy_tokens = [[w for w in s.split() if w not in STOPWORDS_TOY] for s in sentences_toy]
vocab_toy = sorted(set(w for s in toy_tokens for w in s))
word2id_toy = {w: i for i, w in enumerate(vocab_toy)}
print(f"{len(sentences_toy)} sentences, {len(vocab_toy)} content words: {vocab_toy}")

16 sentences, 34 content words: ['applied', 'bank', 'barked', 'canine', 'cat', 'chased', 'deposited', 'dog', 'door', 'feline', 'has', 'hill', 'house', 'interest', 'likes', 'loan', 'loves', 'mailman', 'mat', 'milk', 'money', 'mouse', 'painted', 'park', 'raised', 'rates', 'red', 'river', 'run', 'sat', 'small', 'sold', 'summer', 'warm']


**Idea -- count company, don't assign an axis.** Slide out a window of radius 3 across every sentence. Every time two words fall inside the same window, add one to `cooccurrence[word_i][word_j]`. A word's vector is then simply its row in this table: how many times it fell near every other vocabulary word. Unlike one-hot's private axis, two words that keep the same company will now, mechanically, get similar rows.

In [3]:
def build_cooccurrence(tokenized_sents, vocab, word2id, window):
    M = np.zeros((len(vocab), len(vocab)))
    for sent in tokenized_sents:
        L = len(sent)
        for i, w in enumerate(sent):
            lo, hi = max(0, i - window), min(L, i + window + 1)
            for j in range(lo, hi):
                if j != i:
                    M[word2id[w], word2id[sent[j]]] += 1
    return M

M_toy = build_cooccurrence(toy_tokens, vocab_toy, word2id_toy, window=3)
print("co-occurrence matrix shape:", M_toy.shape)

def cos_toy(a, b):
    va, vb = M_toy[word2id_toy[a]], M_toy[word2id_toy[b]]
    return float(va @ vb / (np.linalg.norm(va) * np.linalg.norm(vb) + 1e-10))

for pair in [("cat", "feline"), ("cat", "dog"), ("cat", "house"), ("cat", "bank")]:
    print(f"cos{pair} = {cos_toy(*pair):.3f}")

co-occurrence matrix shape: (34, 34)
cos('cat', 'feline') = 0.935
cos('cat', 'dog') = 0.000
cos('cat', 'house') = 0.000
cos('cat', 'bank') = 0.000


**Payoff.** `cos(cat, feline) = 0.935` -- a real, non-zero, computed number, versus Slide 4's `dot(cat, feline) = 0`. `cat` and `dog`, `cat` and `house`, `cat` and `bank` all land at exactly `0.000`: in this small corpus those pairs genuinely never share a window, so raw counting has nothing to report about them. That's an honest limitation, not a bug -- counting can only ever see words that were *literally* near each other; it has no way yet to notice that `cat` and `dog` are similar because they occupy *similar*, not identical, neighbourhoods. Keep that gap in mind: it's exactly what a trained model, not a raw count, buys you in Part 2.

**Idea -- see the whole neighbourhood at once.** One pair at a time hides the bigger picture. Turn every pairwise cosine similarity into a heatmap over the full content-word vocabulary and look for blocks -- clusters of mutually similar words -- appearing on their own, with no labels supplied anywhere in this process.

In [4]:
n_toy = len(vocab_toy)
sim_toy = np.zeros((n_toy, n_toy))
for i in range(n_toy):
    for j in range(n_toy):
        vi, vj = M_toy[i], M_toy[j]
        denom = np.linalg.norm(vi) * np.linalg.norm(vj)
        sim_toy[i, j] = (vi @ vj) / denom if denom > 0 else 0.0

fig = go.Figure(go.Heatmap(z=sim_toy, x=vocab_toy, y=vocab_toy,
                            colorscale=[[0, "#fafafa"], [1, GOLD]], zmin=0, zmax=1))
fig.update_layout(title="Cosine similarity between every pair of content words -- no labels used",
                   width=750, height=700)
fig.show()

**Payoff.** Small bright blocks appear exactly where the story predicts them: `cat`/`feline` light up together, `dog`/`canine` light up together, the banking words (`applied`, `deposited`, `loan`, `interest`, `raised`, `rates`) form their own block, and the house words (`door`, `hill`, `painted`, `red`, `summer`, `sold`) form another. Nobody told this matrix which words were synonyms or which sentences were "about" banking -- the blocks are Slide 6's claim, made visible: shared company produces shared geometry, and that geometry was counted, never labelled. Slide 3's five-cluster map was this same phenomenon, just built on millions of words instead of sixteen sentences.

## Part 2 -- Word2Vec from Scratch: Skip-Gram with Negative Sampling

**Problem.** Part 1's rows solved the zero-similarity problem for words that share a literal window, but look at what each row actually is: one number per vocabulary word. Scale that same toy corpus's 34 content words up to a real 50,000-word vocabulary and every row is again 50,000 numbers -- sparse, and blind to word pairs that just never happened to share a window. Slides 9-13 promised a different fix entirely: don't count company, *predict* it. Train a small neural network to guess a word's neighbours, throw the predictions away, and keep the weights -- and use negative sampling (Slide 13) so scoring stays cheap even at real vocabulary size. We want the genuine mechanism, on genuine text, not a toy.

**Ingredients.** The NLTK Gutenberg corpus: eighteen books, tokenized into sentences and lowercased, alphabetic tokens only.

In [5]:
TOKEN_RE = re.compile(r"^[a-z]+$")

sents = []
for fid in gutenberg.fileids():
    for s in gutenberg.sents(fid):
        toks = [w.lower() for w in s if TOKEN_RE.match(w.lower())]
        if len(toks) >= 3:
            sents.append(toks)

n_tokens = sum(len(s) for s in sents)
print(f"{len(sents)} sentences, {n_tokens} tokens, from {len(gutenberg.fileids())} books")

90532 sentences, 2125113 tokens, from 18 books


**Idea -- the architecture, exactly as Slide 10 drew it.** A one-hot input word is looked up as a row of an embedding matrix `W` (`V x D`) -- Slide 10's "multiplying by `W` is just a row lookup." Scoring every candidate neighbour against a second matrix `W'` and running a softmax over all 50,000 words is Slide 13's bottleneck. Negative sampling turns that into a yes/no question per pair: is `coffee` really `morning`'s neighbour? Score it as a dot product `v_center . u_context`, squashed through a sigmoid, and train it toward 1 for real pairs and toward 0 for a handful of randomly sampled fake ones. Written out as a loss to minimise, for one real pair `(c, o)` and `k` sampled negatives `n_1 .. n_k`:

$$
\mathcal{L} = -\log\sigma(v_c \cdot u_o) \;-\; \sum_{i=1}^{k} \log\sigma(-v_c \cdot u_{n_i})
$$

The first term pushes the real pair's dot product up; the second pushes every sampled fake pair's dot product down. Nothing here needs the full 50,000-way softmax -- exactly the "fifteen comparisons instead of fifty thousand" trade Slide 13 described (this lab uses 5, for speed; the shape of the trade is identical).

In [6]:
class SkipGramNegativeSampling(nn.Module):
    """W (in_emb) becomes the word vectors we keep; W' (out_emb) is
    scaffolding, discarded after training -- Slide 10's closing move."""
    def __init__(self, vocab_size, dim):
        super().__init__()
        self.in_emb = nn.Embedding(vocab_size, dim)   # W
        self.out_emb = nn.Embedding(vocab_size, dim)  # W'
        nn.init.uniform_(self.in_emb.weight, -0.5 / dim, 0.5 / dim)
        nn.init.zeros_(self.out_emb.weight)

    def forward(self, center, pos_context, neg_context):
        v_c = self.in_emb(center)                                  # (B, D)   row lookup
        u_pos = self.out_emb(pos_context)                          # (B, D)
        u_neg = self.out_emb(neg_context)                          # (B, K, D)
        pos_score = (v_c * u_pos).sum(-1)                          # v_c . u_o
        neg_score = torch.einsum("bd,bkd->bk", v_c, u_neg)         # v_c . u_n_i, all k at once
        pos_loss = F.logsigmoid(pos_score)
        neg_loss = F.logsigmoid(-neg_score).sum(-1)
        return -(pos_loss + neg_loss).mean()                       # matches the L above

# Sanity check on a tiny random batch before touching real data.
sanity = SkipGramNegativeSampling(vocab_size=100, dim=8)
b = 4
loss_val = sanity(torch.randint(0, 100, (b,)), torch.randint(0, 100, (b,)),
                   torch.randint(0, 100, (b, 5)))
print("sanity-check loss on random data (should be a small positive number, ~log(2)*(1+5) territory):",
      round(loss_val.item(), 3))

sanity-check loss on random data (should be a small positive number, ~log(2)*(1+5) territory): 4.159


**Idea -- pairs for free, and a second, unglamorous fix.** Slide 11's sliding window is exactly how we mine training pairs: for every word in a sentence, every other word within a window of 4 is a real `(center, context)` pair, no labelling required. One extra trick that neither Slide 11 nor Slide 13 covers but every real Word2Vec implementation uses: *downsample* the most frequent words (`the`, `and`, `of`...) the same way Part 1 stripped them by hand -- otherwise a handful of function words dominate almost every window and drown out the words we actually care about. We drop each token with probability that grows the more frequent it is, following the original paper's formula.

In [7]:
counts = Counter()
for s in sents:
    counts.update(s)

MIN_COUNT = 10
vocab = [w for w, c in counts.items() if c >= MIN_COUNT]
vocab_size = len(vocab)
word2id = {w: i for i, w in enumerate(vocab)}
print("vocabulary (words appearing >= 10 times):", vocab_size)

total_count = sum(counts[w] for w in vocab)
T = 1e-4  # standard Word2Vec subsampling threshold

def keep_probability(w):
    f = counts[w] / total_count
    return min(1.0, (np.sqrt(f / T) + 1) * (T / f))

keep_p = {w: keep_probability(w) for w in vocab}

rng = np.random.default_rng(0)
WINDOW = 4
centers, contexts = [], []
for s in sents:
    ids = [word2id[w] for w in s if w in word2id and rng.random() < keep_p[w]]
    L = len(ids)
    for i, c in enumerate(ids):
        lo, hi = max(0, i - WINDOW), min(L, i + WINDOW + 1)
        for j in range(lo, hi):
            if j != i:
                centers.append(c)
                contexts.append(ids[j])

centers = np.array(centers, dtype=np.int64)
contexts = np.array(contexts, dtype=np.int64)
print(f"{len(centers)} training pairs after subsampling (window={WINDOW})")

vocabulary (words appearing >= 10 times): 9957


5388422 training pairs after subsampling (window=4)


**Idea -- billions of nudges, four epochs' worth.** Slide 50 called training "billions of nudges, across billions of positions." We have millions, not billions, of pairs -- but the loop is the same loop: draw a batch of real pairs plus fresh random negatives (sampled from a frequency-weighted noise distribution, per Slide 13), compute the loss above, and step. Track the average loss every epoch; if the mechanism is actually learning, it should fall, the same falling-curve check the k-means lab ran on WCSS.

In [8]:
freqs = np.array([counts[w] for w in vocab], dtype=np.float64)
noise_dist = torch.tensor((freqs ** 0.75) / (freqs ** 0.75).sum(), dtype=torch.float64)

DIM, NEG, EPOCHS, BATCH, LR = 100, 5, 4, 4096, 0.003
torch.manual_seed(0)  # re-seed here so the sanity check above can't shift these draws
model = SkipGramNegativeSampling(vocab_size, DIM)
opt = torch.optim.Adam(model.parameters(), lr=LR)

centers_t = torch.tensor(centers, dtype=torch.long)
contexts_t = torch.tensor(contexts, dtype=torch.long)
n_pairs = len(centers)

loss_history = []
t_start = time.time()
for epoch in range(EPOCHS):
    perm = torch.randperm(n_pairs)
    total_loss, n_batches = 0.0, 0
    for start in range(0, n_pairs, BATCH):
        idx = perm[start:start + BATCH]
        neg = torch.multinomial(noise_dist, len(idx) * NEG, replacement=True).view(len(idx), NEG)
        loss = model(centers_t[idx], contexts_t[idx], neg)
        opt.zero_grad()
        loss.backward()
        opt.step()
        total_loss += loss.item()
        n_batches += 1
    avg_loss = total_loss / n_batches
    loss_history.append(avg_loss)
    print(f"epoch {epoch}: avg loss {avg_loss:.4f}  ({time.time() - t_start:.0f}s elapsed)")

W = model.in_emb.weight.detach().numpy()  # keep W; W' is scaffolding, same as Slide 10's closing move

fig = go.Figure(go.Scatter(y=loss_history, mode="lines+markers", line=dict(color=GOLD)))
fig.update_layout(title="Negative-sampling loss by epoch", xaxis_title="epoch",
                   yaxis_title="avg loss per pair", width=650, height=400)
fig.show()

assert loss_history[-1] < loss_history[0], "loss should fall"

epoch 0: avg loss 2.5873  (33s elapsed)


epoch 1: avg loss 2.3804  (66s elapsed)


epoch 2: avg loss 2.3114  (98s elapsed)


epoch 3: avg loss 2.2665  (173s elapsed)


**Payoff.** The loss falls every single epoch (from about 2.59 down to about 2.27, exact numbers above) and the assert would have failed loudly otherwise -- the mechanism is genuinely learning something, on real text, in a few minutes, entirely without labels. But notice what the loss curve does *not* tell you: whether what it learned is any *good*. That's a different question, and it's the one Part 3 actually asks.

## Part 3 -- Explore the Space: Neighbours and Analogies

**Problem.** Slide 14 claimed the weights learn semantic similarity, syntax, world knowledge, and bias. Slide 15 paid off the whole session's opening hook: `king - man + woman` should land near `queen`. Those are claims about *this* model's weights -- not yet checked against them. Let's check.

**Ingredients.** The trained matrix `W` from Part 2 and its `word2id` lookup -- nothing new to load.

In [9]:
def most_similar(word, topn=8):
    v = W[word2id[word]]
    norms = np.linalg.norm(W, axis=1) * np.linalg.norm(v) + 1e-10
    sims = (W @ v) / norms
    order = np.argsort(-sims)
    out = [(vocab[i], round(float(sims[i]), 3)) for i in order if vocab[i] != word][:topn]
    return out

**Idea -- nearest neighbour is just the highest cosine similarity in the row space.** Exactly Slide 14's claim, made computable: pick a word, find the other rows of `W` its own row points closest to, and read off whether the neighbourhood looks like a *meaning* neighbourhood.

In [10]:
for w in ["king", "queen", "water", "bank"]:
    print(f"{w:8s} -> {most_similar(w, 6)}")

king     -> [('riblah', 0.699), ('zedekiah', 0.695), ('darius', 0.686), ('sennacherib', 0.68), ('jehoiakim', 0.673), ('ahaz', 0.672)]
queen    -> [('vashti', 0.598), ('esther', 0.58), ('ahasuerus', 0.545), ('banquet', 0.522), ('chamberlains', 0.503), ('mordecai', 0.492)]
water    -> [('thirsty', 0.624), ('washed', 0.593), ('dry', 0.572), ('drink', 0.566), ('wash', 0.552), ('pools', 0.527)]
bank     -> [('mud', 0.605), ('gilt', 0.582), ('aroer', 0.578), ('river', 0.577), ('pool', 0.576), ('trundle', 0.55)]


**Payoff.** `king`'s nearest neighbours are `zedekiah`, `darius`, `sennacherib`, `jehoiakim`, `ahaz` -- real named biblical kings, not a generic notion of "royalty." `queen`'s neighbours include `vashti`, `esther`, `ahasuerus`, and, at rank 7, `king` itself. `bank`'s neighbours (`river`, `pool`, `mud`) are entirely the riverbank sense -- unsurprising, since this corpus is 19th-century-and-earlier literary and religious text with essentially no discussion of financial institutions (Slide 20's polysemy problem is still three slides away, but its "one blurred vector" villain is quietly on display here already: `bank` got exactly one vector, and this corpus only ever fed it one sense to blur). Takeaway: nearest-neighbour geometry reflects the training *text's* own world, not a universal notion of meaning -- a corpus of scripture and 19th-century novels produces a different neighbourhood for `king` than a corpus of modern news would.

**Idea -- analogy arithmetic is the same vector offset Slide 15 drew by hand.** `a - b + c`, then find whichever word's row sits closest to that resulting point, excluding the three query words themselves.

In [11]:
def analogy(a, b, c, topn=6):
    v = W[word2id[a]] - W[word2id[b]] + W[word2id[c]]
    norms = np.linalg.norm(W, axis=1) * np.linalg.norm(v) + 1e-10
    sims = (W @ v) / norms
    order = np.argsort(-sims)
    exclude = {a, b, c}
    return [(vocab[i], round(float(sims[i]), 3)) for i in order if vocab[i] not in exclude][:topn]

print("brother - man + woman ->", analogy("brother", "man", "woman"))
print("husband - man + woman ->", analogy("husband", "man", "woman"))
print("king    - man + woman ->", analogy("king", "man", "woman"))

brother - man + woman -> [('sister', 0.64), ('fanny', 0.566), ('wife', 0.553), ('husband', 0.552), ('nurse', 0.53), ('sarai', 0.518)]
husband - man + woman -> [('wife', 0.593), ('married', 0.549), ('sister', 0.529), ('hannah', 0.528), ('marry', 0.523), ('mother', 0.523)]
king    - man + woman -> [('bathsheba', 0.617), ('eunuchs', 0.609), ('rabshakeh', 0.555), ('sennacherib', 0.542), ('assyria', 0.532), ('ishbosheth', 0.529)]


**Payoff.** `brother - man + woman` lands on `sister` at rank 1. `husband - man + woman` lands on `wife` at rank 1. Both are the parallelogram trick, genuinely working, on vectors nobody hand-tuned. But `king - man + woman`, the session's own opening hook, does *not* return `queen` anywhere in its top 6 -- `queen` turns out to rank 95th. The exact demo this session opened with does not reproduce on this corpus. Takeaway: Slide 15's own caution -- "plenty of analogies come out fuzzier" -- turns out to be true of literally the showpiece example itself, once you leave the large modern web corpus Word2Vec's original paper trained on. That honest gap is exactly what the Challenge investigates.

## Part 4 -- Visualise with t-SNE

**Problem.** Slide 16 showed a five-cluster map (animals, countries, royalty, food, numbers) and an honesty note alongside it: t-SNE preserves *within*-cluster distance reasonably well but not the *gaps between* clusters -- "trust the huddles, not the gaps." That claim, so far, is just the lecturer's word for it. Let's build the actual picture on our own vectors, and put a number behind the warning instead of only eyeballing it.

**Ingredients.** Five category word lists in the same spirit as Slide 3's map, chosen for what this particular corpus actually contains in quantity (kinship and royalty terms are everywhere in Gutenberg's scripture and Shakespeare; modern categories like "countries" or "tech brands" would mostly be missing words).

In [12]:
categories = {
    "kinship": ["man", "woman", "father", "mother", "son", "daughter",
                "brother", "sister", "husband", "wife"],
    "royalty": ["king", "queen", "prince", "throne", "crown", "kingdom"],
    "nature":  ["water", "river", "mountain", "sea", "wind", "fire", "earth", "wilderness"],
    "numbers": ["one", "two", "three", "four", "five", "seven", "ten", "hundred"],
    "animals": ["sheep", "lamb", "ox", "lion", "dove", "serpent", "horse", "ass"],
}
missing = [w for ws in categories.values() for w in ws if w not in word2id]
assert not missing, f"missing from vocab: {missing}"

cat_colors = {"kinship": GOLD, "royalty": BLUE, "nature": TEAL, "numbers": RED, "animals": DIM}
words_flat = [w for ws in categories.values() for w in ws]
labels_flat = [cat for cat, ws in categories.items() for _ in ws]
print(f"{len(words_flat)} words across {len(categories)} categories, all present in vocab")

40 words across 5 categories, all present in vocab


**Idea -- project 100 dimensions down to 2, and look for the map from Slide 3.** t-SNE arranges points so that vectors close together in the real 100-dimensional space stay close on the page. Colour each point by its category and see whether the categories separate visually, exactly the picture Slide 16 showed for the lecture's own (much larger) trained space.

In [13]:
coords = TSNE(n_components=2, perplexity=8, random_state=0,
              init="pca").fit_transform(W[[word2id[w] for w in words_flat]])

fig = go.Figure()
for cat in categories:
    idx = [i for i, l in enumerate(labels_flat) if l == cat]
    fig.add_trace(go.Scatter(x=coords[idx, 0], y=coords[idx, 1], mode="markers+text",
                              text=[words_flat[i] for i in idx], textposition="top center",
                              marker=dict(color=cat_colors[cat], size=10), name=cat))
fig.update_layout(title="Five categories, projected from 100 dimensions to 2 -- no labels used in training",
                   width=800, height=550)
fig.show()

**Payoff.** `numbers` and `kinship` each pull into their own tight, separate huddle; `royalty` and `animals` sit as visible groups too, though `nature` bleeds a little into neighbouring groups (`fire`, `earth`, and `wilderness` drift toward different neighbours) -- a real, imperfect result, not a cherry-picked clean one. The categories were never shown to the model; they only exist because we chose to colour the points that way afterwards.

**Idea -- put a number behind "trust the huddles."** A picture alone can't tell you whether the *gaps between* clusters mean anything, per Slide 16's warning. Compare, in the real 100-dimensional space (not the 2D picture), the average cosine similarity between two words in the *same* category against the average between two words in *different* categories. If Slide 16's claim about within-cluster distance is right, the first number should clearly beat the second.

In [14]:
def cos(a, b):
    va, vb = W[word2id[a]], W[word2id[b]]
    return float(va @ vb / (np.linalg.norm(va) * np.linalg.norm(vb) + 1e-10))

within, between = [], []
for i in range(len(words_flat)):
    for j in range(i + 1, len(words_flat)):
        c = cos(words_flat[i], words_flat[j])
        (within if labels_flat[i] == labels_flat[j] else between).append(c)

print(f"within-category mean cosine:  {np.mean(within):.3f}  (n={len(within)} pairs)")
print(f"between-category mean cosine: {np.mean(between):.3f}  (n={len(between)} pairs)")

within-category mean cosine:  0.404  (n=144 pairs)
between-category mean cosine: 0.154  (n=636 pairs)


**Payoff.** Within-category pairs average `0.404`; between-category pairs average `0.154` -- a real, sizeable gap, computed in the original 100-dimensional space where t-SNE never touched anything. Slide 16's warning holds up under a number, not just a picture: trust that same-category words huddle together (they measurably do, in the space that matters), but the specific *distances between cluster centres* in the 2D picture above are a t-SNE layout choice, not something you can read a number out of directly.

## Challenge -- Break It: Which Analogies Fail, and Why

**Problem.** Slide 15 warned that the parallelogram trick "works beautifully for [the showpiece example]; plenty of analogies come out fuzzier." Part 3 already found one real failure -- `king - man + woman` missed `queen` by 94 places. Time to go looking for more failures on purpose, and actually diagnose *why* each one breaks: a data problem, a window problem, or the polysemy wall itself (Slide 20), rather than shrugging at a wrong answer.

**Ingredients.** Six analogy tests spanning royalty, kinship, and verb tense, plus the raw word-frequency counts we'll need to diagnose whatever breaks.

In [15]:
def analogy_rank(a, b, c, target, cap=200):
    """Where does `target` actually land in the sorted analogy result?
    A rank of 1 is a clean hit; a rank in the hundreds is a clean miss."""
    v = W[word2id[a]] - W[word2id[b]] + W[word2id[c]]
    norms = np.linalg.norm(W, axis=1) * np.linalg.norm(v) + 1e-10
    sims = (W @ v) / norms
    order = np.argsort(-sims)
    exclude = {a, b, c}
    rank = 0
    for i in order:
        if vocab[i] in exclude:
            continue
        rank += 1
        if vocab[i] == target:
            return rank
        if rank > cap:
            return None

tests = [
    ("king", "man", "woman", "queen"),
    ("brother", "man", "woman", "sister"),
    ("husband", "man", "woman", "wife"),
    ("father", "man", "woman", "mother"),
    ("son", "man", "woman", "daughter"),
    ("walked", "walk", "run", "ran"),
]

**Idea -- score every test with a rank, not a top-1 guess.** Eyeballing whether the #1 result "looks right" hides how close a near-miss actually was. Call anything ranked 5th or better a hit; anything worse (including "not found in the top 200") a miss, and tabulate the whole battery.

In [16]:
print(f"{'analogy':38s}{'target':10s}{'rank':>6s}   verdict")
for a, b, c, target in tests:
    rank = analogy_rank(a, b, c, target)
    verdict = "HIT " if rank is not None and rank <= 5 else "MISS"
    rank_str = str(rank) if rank is not None else ">200"
    print(f"{a+' - '+b+' + '+c:38s}{target:10s}{rank_str:>6s}   {verdict}")

analogy                               target      rank   verdict
king - man + woman                    queen         95   MISS
brother - man + woman                 sister         1   HIT 
husband - man + woman                 wife           1   HIT 
father - man + woman                  mother         1   HIT 
son - man + woman                     daughter      46   MISS
walked - walk + run                   ran           18   MISS


**Payoff.** Three clean hits (`brother`/`sister`, `husband`/`wife`, `father`/`mother`, all rank 1) and three misses (`king`/`queen` at rank 95, `son`/`daughter` at rank 46, `walked`/`ran` at rank 18). Notice that "kinship analogy" alone doesn't predict success: `father`/`mother` is a clean hit, but `son`/`daughter` -- the *same* broad category -- is a clear miss. Something specific broke each failing case differently, not one shared cause. Worth digging into which.

**Idea -- test the "specific cause" hypothesis with real corpus evidence.** Two candidate explanations for `king`/`queen`'s failure: `king` (3,037 occurrences) is far more frequent than `queen` (198 occurrences) in this corpus, giving `queen` a noisier, less-trained vector; and/or `king`'s nearest neighbours (Part 3) were specific named monarchs, not a generic royalty role. For `son`/`daughter`, a different hypothesis: biblical genealogy repeats the fixed phrase "son of X" ("the son of Jesse," "son of man"...) far more than any equivalent "daughter of" phrasing, which could pull `son`'s vector toward that formula instead of a generic "male child" role. Check both, with counts, not guesses.

In [17]:
def phrase_ratio(word, next_word):
    total = sum(1 for s in sents for w in s if w == word)
    followed = sum(1 for s in sents for i, w in enumerate(s)
                    if w == word and i + 1 < len(s) and s[i + 1] == next_word)
    return total, followed, followed / total

for word in ("king", "queen"):
    print(f"freq({word}) = {counts[word]}")

for word in ("son", "daughter"):
    total, followed, ratio = phrase_ratio(word, "of")
    print(f'"{word}" occurs {total} times; followed by "of" {followed} times ({ratio:.1%})')

freq(king) = 3037
freq(queen) = 198


"son" occurs 2738 times; followed by "of" 1651 times (60.3%)


"daughter" occurs 572 times; followed by "of" 195 times (34.1%)


**Payoff -- your answer (2-3 sentences).** Look at the numbers just printed: `king` occurs roughly 15x more often than `queen`, and `king`'s own nearest neighbours (Part 3) were named monarchs, not a generic role. `son` is followed by "of" roughly 60% of the time, against `daughter`'s roughly 34%. Using those real numbers, and the narration's own three-way framing (a data problem, a window problem, or the polysemy wall itself), diagnose *both* failures in your own words -- which cause fits which failure, and why doesn't the same cause explain both?

*(write your answer here)*

## Reflection

**Problem.** You now own the whole arc: counting company by hand, training the actual Skip-Gram mechanism from its two matrices up, checking its neighbours and analogies against the lecture's own claims, and diagnosing exactly where and why the session's opening hook fails to reproduce. The point of this last cell isn't a right answer -- it's stating, in your own words, the reasoning each part was built on.

In [18]:
# 1. Part 1 found cos(cat, feline) = 0.935 but cos(cat, dog) = cos(cat, house) =
#    cos(cat, bank) = 0.000 exactly. Part 3's TRAINED model, by contrast, found
#    real neighbours for "king" and "queen" that never once appear in the same
#    sentence as each other in this corpus. What can raw co-occurrence counting
#    never capture that a TRAINED model can -- and why?
answer_1 = """
"""

# 2. Part 2's loss fell every epoch (about 2.59 down to about 2.27), which the
#    assert confirmed. Part 3 then found that king - man + woman completely
#    misses queen (rank 95). Put these two findings together: what does a
#    falling training loss guarantee about a model, and what does it NOT
#    guarantee?
answer_2 = """
"""

# 3. Part 4 found within-category cosine (0.404) clearly above between-category
#    cosine (0.154), computed in the real 100-dimensional space. Explain, in
#    your own words, why that number was necessary -- why couldn't the t-SNE
#    PICTURE alone have told you whether Slide 16's "trust the huddles, not the
#    gaps" warning was actually true?
answer_3 = """
"""

# 4. The Challenge found "son" followed by "of" about 60% of the time versus
#    "daughter" about 34% of the time, and king (3,037 occurrences) vastly
#    outnumbering queen (198). Using the narration's own three-way framing --
#    a data problem, a window problem, or the polysemy wall itself (Slide 20)
#    -- which category does EACH of the two failures (king/queen, son/daughter)
#    belong to, and why aren't they the same category?
answer_4 = """
"""

# 5. This whole lab trained on 2.1 million words of 19th-century-and-earlier
#    literary and religious English, not the billions of words of modern web
#    text the original Word2Vec paper used. If you reran this EXACT code on a
#    large modern news corpus instead, what specifically would you predict
#    changes about the king - man + woman result, and why?
answer_5 = """
"""

---
*Words as Vectors: From One-Hot to Geometry · Bluevision-ai Academy*